# CardioFusion-AI -- Quickstart Demo

This notebook demonstrates the ECG/PPG preprocessing and synchronization pipeline end-to-end using **synthetic signals**, since no real dataset ships with this repository (see `datasets/README.md`).

Swap `synth_ecg` / `synth_ppg` below for real recordings loaded via `training/dataset.py` once you have PhysioNet/PulseDB/BIDMC data downloaded, and the rest of the pipeline runs unchanged.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from preprocessing.ecg.ecg_preprocessing import ECGProcessingConfig, extract_ecg_pipeline
from preprocessing.ppg.ppg_preprocessing import PPGProcessingConfig, extract_ppg_pipeline
from preprocessing.synchronization.sync import synchronize
from visualization.plots import plot_ecg_ppg_sync, plot_ecg_with_rpeaks, plot_ppg_with_peaks

In [ ]:
def synth_ecg(duration=20, fs=250, hr_bpm=72, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / fs)
    beat_interval = 60 / hr_bpm
    ecg = np.zeros_like(t)
    for bt in np.arange(0, duration, beat_interval):
        idx = int(bt * fs)
        if idx < len(ecg):
            w = int(0.02 * fs)
            for i in range(max(0, idx - w), min(len(ecg), idx + w)):
                ecg[i] += np.exp(-0.5 * ((i - idx) / (w / 3)) ** 2) * 3.0
    ecg += 0.05 * np.sin(2 * np.pi * 0.3 * t) + 0.02 * np.sin(2 * np.pi * 50 * t)
    ecg += rng.normal(0, 0.05, len(t))
    return ecg, fs

def synth_ppg(duration=20, fs=100, hr_bpm=72, ptt_offset=0.2, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(0, duration, 1 / fs)
    beat_interval = 60 / hr_bpm
    ppg = np.zeros_like(t)
    for bt in np.arange(ptt_offset, duration, beat_interval):
        idx = int(bt * fs)
        for i in range(len(t)):
            rel = (i - idx) / fs
            if -0.1 <= rel < 0.5:
                ppg[i] += np.exp(-0.5 * (rel / 0.04) ** 2) if rel < 0 else np.exp(-rel / 0.15)
    ppg += 3.0 + 0.03 * np.sin(2 * np.pi * 0.2 * t) + rng.normal(0, 0.015, len(t))
    return ppg, fs

raw_ecg, fs_ecg = synth_ecg()
raw_ppg, fs_ppg = synth_ppg()
print(f"ECG: {len(raw_ecg)} samples @ {fs_ecg}Hz | PPG: {len(raw_ppg)} samples @ {fs_ppg}Hz")

## Preprocess each modality

In [ ]:
clean_ecg, ecg_features = extract_ecg_pipeline(raw_ecg, ECGProcessingConfig(fs=fs_ecg))
clean_ppg, ppg_features = extract_ppg_pipeline(raw_ppg, PPGProcessingConfig(fs=fs_ppg))

print(f"ECG heart rate: {ecg_features.heart_rate_bpm:.1f} bpm | SDNN: {ecg_features.hrv_sdnn:.1f} ms")
print(f"PPG pulse rate: {ppg_features.pulse_rate_bpm:.1f} bpm | perfusion index: {ppg_features.perfusion_index:.2f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
plot_ecg_with_rpeaks(clean_ecg[:fs_ecg*8], ecg_features.r_peaks[ecg_features.r_peaks < fs_ecg*8], fs_ecg, ax=axes[0])
plot_ppg_with_peaks(clean_ppg[:fs_ppg*8], ppg_features.systolic_peaks[ppg_features.systolic_peaks < fs_ppg*8], fs_ppg, ax=axes[1])
fig.tight_layout()
plt.show()

## Synchronize ECG and PPG

Estimates the Pulse Transit Time (PTT) via cross-correlation and aligns both streams onto a common timeline -- required before feeding them into `EarlyFusion`, and helpful context even for the looser fusion strategies.

In [ ]:
sync_result = synchronize(clean_ecg, fs_ecg, clean_ppg, fs_ppg)
print(f"Estimated PTT: {sync_result.estimated_ptt_ms:.1f} ms | common fs: {sync_result.common_fs} Hz")

fig = plot_ecg_ppg_sync(
    sync_result.ecg_resampled[: sync_result.common_fs * 6],
    sync_result.ppg_resampled[: sync_result.common_fs * 6],
    sync_result.common_fs,
    ptt_ms=sync_result.estimated_ptt_ms,
)
plt.show()

## Next steps

1. Replace the synthetic generators above with real recordings loaded via `training/dataset.py`'s dataset-specific loaders.
2. Cache cleaned/windowed arrays to `datasets/cache/`.
3. Run `python -m training.train --config configs/default.yaml` (requires `torch` -- not available in this sandbox but is a standard `pip install torch`).
4. Evaluate with `evaluation/evaluate.py` and inspect explainability outputs via `models/explainability/explain.py`.